# C-MAPSS 터보팬 엔진 RUL 예측 - 전체 실험 재현

데이터 다운로드부터 학습/평가까지 이 노트북 하나로 재현 가능.

- 런타임 > 런타임 유형 변경 > **GPU(T4)** 권장 (CPU로도 10분 내외)
- 마지막 셀에서 결과 그림/지표를 zip으로 다운로드 → repo의 `results/`에 복사


In [ ]:
import kagglehub, os, glob

path = kagglehub.dataset_download("behrad3d/nasa-cmaps")
print(path)

# 데이터셋 폴더 구조가 바뀔 수 있어서 재귀로 찾음
train_path = glob.glob(os.path.join(path, "**", "train_FD001.txt"), recursive=True)[0]
data_dir = os.path.dirname(train_path)
print(data_dir, os.listdir(data_dir)[:8])


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

COLS = ['unit', 'cycle', 'set1', 'set2', 'set3'] + [f's{i}' for i in range(1, 22)]

def load_txt(p):
    df = pd.read_csv(p, sep=r'\s+', header=None)
    df.columns = COLS
    return df

train = load_txt(os.path.join(data_dir, 'train_FD001.txt'))
test = load_txt(os.path.join(data_dir, 'test_FD001.txt'))
rul_true = pd.read_csv(os.path.join(data_dir, 'RUL_FD001.txt'),
                       sep=r'\s+', header=None)[0].to_numpy(np.float32)
print(train.shape, test.shape, rul_true.shape)
train.head(3)


## 1. EDA

- 엔진별 수명 분포
- 센서가 고장에 가까워질수록 어떻게 변하는지
- 값이 아예 안 변하는 센서 확인 (FD001은 운전조건이 하나라 상수 센서가 있음)


In [ ]:
os.makedirs("results_export", exist_ok=True)

life = train.groupby('unit')['cycle'].max()
print(f"엔진 수명: min {life.min()} / median {life.median():.0f} / max {life.max()}")

# 상수 센서 확인
stds = train[[f's{i}' for i in range(1, 22)]].std()
print("\nstd가 거의 0인 센서:")
print(stds[stds < 0.01])


In [ ]:
# 엔진 1대의 센서 몇 개를 수명 전체에 걸쳐 보기
u1 = train[train['unit'] == 1]
show = ['s2', 's3', 's4', 's7', 's11', 's15']
fig, axes = plt.subplots(2, 3, figsize=(13, 5))
for ax, s in zip(axes.flat, show):
    ax.plot(u1['cycle'], u1[s])
    ax.set_title(s, fontsize=9)
fig.suptitle('unit 1 sensor trends (run to failure)')
fig.tight_layout()
fig.savefig("results_export/sensor_trends.png", dpi=150)
plt.show()
# 고장(오른쪽 끝)에 가까워질수록 드리프트가 생기는 게 보임


## 2. 전처리

repo의 `src/prepare_data.py`와 동일한 로직.

- 상수 센서 7개 제외 → 14개 사용
- min-max 정규화 (train 기준으로만 산출)
- RUL 상한 125로 클리핑
- 슬라이딩 윈도우 30
- train/val은 엔진(unit) 단위로 분할 (윈도우 단위로 나누면 누설)


In [ ]:
DROP = ['s1', 's5', 's6', 's10', 's16', 's18', 's19']
FEATURES = [c for c in [f's{i}' for i in range(1, 22)] if c not in DROP]
WINDOW, CLIP, SEED = 30, 125, 42

f_min = train[FEATURES].min().to_numpy(np.float32)
f_max = train[FEATURES].max().to_numpy(np.float32)
rng_ = np.where(f_max - f_min == 0, 1, f_max - f_min)
for df in (train, test):
    df[FEATURES] = (df[FEATURES] - f_min) / rng_

def make_windows(df):
    Xs, ys, units = [], [], []
    for uid, g in df.groupby('unit'):
        arr = g[FEATURES].to_numpy(np.float32)
        max_cycle = g['cycle'].max()
        if len(arr) < WINDOW:
            arr = np.vstack([np.repeat(arr[:1], WINDOW - len(arr), 0), arr])
        for end in range(WINDOW, len(arr) + 1):
            Xs.append(arr[end - WINDOW:end])
            rul = max_cycle - g['cycle'].iloc[min(end, len(g)) - 1]
            ys.append(min(rul, CLIP))
            units.append(uid)
    return np.stack(Xs), np.array(ys, np.float32), np.array(units)

def last_window(df):
    Xs = []
    for _, g in df.groupby('unit'):
        arr = g[FEATURES].to_numpy(np.float32)
        if len(arr) < WINDOW:
            arr = np.vstack([np.repeat(arr[:1], WINDOW - len(arr), 0), arr])
        Xs.append(arr[-WINDOW:])
    return np.stack(Xs)

X, y, units = make_windows(train)
rng = np.random.default_rng(SEED)
val_ids = rng.choice(np.unique(units), 20, replace=False)
m = np.isin(units, val_ids)
X_tr, y_tr, X_val, y_val = X[~m], y[~m], X[m], y[m]
X_te, y_te = last_window(test), np.minimum(rul_true, CLIP)
print(X_tr.shape, X_val.shape, X_te.shape)


## 3. 학습

repo의 `src/model.py`, `src/train.py`와 동일한 코드.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import time, random

class RulLSTM(nn.Module):
    def __init__(self, n_features=14, hidden=64, layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, num_layers=layers,
                            batch_first=True, dropout=dropout)
        self.head = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(hidden, 32),
            nn.ReLU(inplace=True), nn.Linear(32, 1))

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1]).squeeze(-1)

def set_seed(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)

@torch.no_grad()
def rmse_of(model, loader, device):
    model.eval()
    se, n = 0.0, 0
    for xb, yb in loader:
        pred = model(xb.to(device))
        se += ((pred.cpu() - yb) ** 2).sum().item(); n += len(yb)
    return (se / n) ** 0.5

set_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

train_loader = DataLoader(TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr)),
                          batch_size=256, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val)),
                        batch_size=512)

model = RulLSTM(n_features=len(FEATURES)).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 30

history = {'train_loss': [], 'val_rmse': []}
best = float('inf')
for epoch in range(EPOCHS):
    model.train(); t0 = time.time(); losses = []
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward(); optimizer.step()
        losses.append(loss.item())
    vr = rmse_of(model, val_loader, device)
    history['train_loss'].append(float(np.mean(losses)))
    history['val_rmse'].append(float(vr))
    mark = ''
    if vr < best:
        best = vr
        torch.save(model.state_dict(), 'best_model.pt')
        mark = ' *'
    print(f"[{epoch+1:2d}/{EPOCHS}] loss {np.mean(losses):.1f} | val RMSE {vr:.2f} | {time.time()-t0:.0f}s{mark}")


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(history['train_loss']); ax[0].set_title('train loss (MSE)'); ax[0].set_xlabel('epoch')
ax[1].plot(history['val_rmse']); ax[1].set_title('val RMSE (cycles)'); ax[1].set_xlabel('epoch')
fig.tight_layout()
fig.savefig("results_export/training_curve.png", dpi=150)
plt.show()


## 4. 테스트셋 평가 (엔진 100대의 마지막 시점 RUL)


In [ ]:
model.load_state_dict(torch.load('best_model.pt'))
model.eval()
with torch.no_grad():
    pred = model(torch.from_numpy(X_te).to(device)).cpu().numpy()

rmse = float(np.sqrt(np.mean((pred - y_te) ** 2)))
mae = float(np.mean(np.abs(pred - y_te)))
print(f"test RMSE: {rmse:.2f} cycles")
print(f"test MAE : {mae:.2f} cycles")
with open("results_export/metrics.txt", "w") as f:
    f.write(f"test RMSE: {rmse:.2f}\ntest MAE: {mae:.2f}\n")

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(y_te, pred, s=18, alpha=0.7)
lim = max(y_te.max(), pred.max()) + 10
ax.plot([0, lim], [0, lim], 'r--', lw=1)
ax.set_xlabel('true RUL (cycles)'); ax.set_ylabel('predicted RUL (cycles)')
ax.set_title(f'test units (n={len(y_te)}), RMSE={rmse:.1f}')
fig.tight_layout()
fig.savefig("results_export/pred_vs_true.png", dpi=150)
plt.show()


In [ ]:
order = np.argsort(y_te)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(y_te[order], label='true')
ax.plot(pred[order], label='pred', alpha=0.8)
ax.set_xlabel('test unit (sorted by true RUL)'); ax.set_ylabel('RUL (cycles)')
ax.legend()
fig.tight_layout()
fig.savefig("results_export/per_unit.png", dpi=150)
plt.show()
# RUL이 작은 구간(왼쪽, 고장 임박)이 잘 맞는지가 중요


In [ ]:
import shutil
shutil.make_archive("results_export", "zip", "results_export")
from google.colab import files
files.download("results_export.zip")


---

받은 zip의 그림을 repo `results/`에 넣고, `metrics.txt`의 RMSE/MAE를
README 결과 표에 기입하면 끝.
